# S2.5 — Narrow vs Wide Transformations
**Date completed:** May 2025  
**Status:** In Progress  
**Interview covered:** Q14 — Narrow vs Wide transformations

In [0]:
import time
import pyspark.sql.functions  as F

# Narrow transformation proof

In [0]:
print("=== NARROW TRANSFORMATION ===")
print("filter() - data stays on same exedcutor, no shuffle")
print()

df = spark.range(0, 5000000)

start = time.time()
result = df.filter(df.id <2500000).count()
end = time.time()


print(f"Result: {result} rows")
print(f"Time taken: {round(end - start, 4)} seconds")
print("Check Query Profile - No shuffle step appears")

In [0]:
print("=== WIDE TRANSFORMATION ===")
print("groupBy() — data moves between executors, shuffle required")
print()

df = spark.range(0, 5000000)
df = df.withColumn("city", 
    F.when(df.id % 3 == 0, "Delhi")
    .when(df.id % 3 == 1, "Mumbai")
    .otherwise("Chennai"))

start = time.time()
result = df.groupBy("city").count()
result.show()
end = time.time()

print(f"Time taken: {round(end - start, 4)} seconds")
print("Check Query Profile — shuffle step appears with LARGE data movement")

## Key Takeaways — S2.5

## Narrow vs Wide — One Rule
Ask: "Can this executor complete this using ONLY its own rows?"
- YES = NARROW (no shuffle)
- NO  = WIDE (shuffle required)

## Examples
| Transformation | Type | Shuffle |
|---------------|------|---------|
| filter() | Narrow | No — data stays on executor |
| select() | Narrow | No — column selection is local |
| withColumn() | Narrow | No — row-level operation |
| when/otherwise | Narrow | No — if/else per row |
| groupBy() | Wide | YES — all same-key rows must meet |
| join() | Wide | YES — matching rows from both sides |
| distinct() | Wide | YES — needs all rows to deduplicate |

## Query Profile Proof
| | Narrow (filter) | Wide (groupBy) |
|--|----------------|---------------|
| Shuffle rows | 8 rows | 24 rows |
| Most expensive step | Filter 6ms | GroupBy Aggregate 131ms |

## Spark is smart
groupBy does NOT shuffle all 5M rows
Partial aggregation first → only 24 summary rows travel